# Example Usage - Econometrics Utils

This notebook demonstrates how to use the custom utility functions for Wooldridge exercises.

---

## 1. Setup and Imports

In [2]:
# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Custom utilities
import sys
sys.path.append('..')
from utils import (
    load_wooldridge_data,
    run_ols,
    run_ols_formula,
    pretty_print_results,
    check_heteroskedasticity,
    check_multicollinearity,
    durbin_watson_test,
    plot_residuals,
    plot_correlation_matrix,
    summary_statistics,
    check_gpu_available
)

# Check GPU availability
print(f"GPU Available: {check_gpu_available()}")

GPU Available: False


## 2. Loading Data

The `load_wooldridge_data()` function makes it easy to load datasets.

**Note:** Make sure you've downloaded the dataset first using `download_data.py`

In [ ]:
# Example: Load the wage1 dataset
# First, run: python download_data.py
# Or manually download wage1.xls from the dataset website

# df = load_wooldridge_data('wage1')
# df.head()

# For now, let's create sample data to demonstrate
np.random.seed(42)
n = 500
df = pd.DataFrame({
    'wage': 5 + 0.5 * np.random.randn(n) ** 2 + np.random.randn(n) * 2,
    'educ': np.random.randint(8, 20, n),
    'exper': np.random.randint(0, 40, n),
    'tenure': np.random.randint(0, 20, n)
})

# Add some relationship
df['wage'] = 2 + 0.5 * df['educ'] + 0.1 * df['exper'] + 0.2 * df['tenure'] + np.random.randn(n) * 2

print("Sample data created for demonstration")
df.head()

## 3. Summary Statistics

Quick overview of your data:

In [ ]:
summary_statistics(df)

## 4. Correlation Analysis

In [ ]:
plot_correlation_matrix(df)

## 5. Running OLS Regression

### Method 1: Using variable lists

In [ ]:
# Run OLS regression
results = run_ols(
    data=df,
    dependent='wage',
    independent=['educ', 'exper', 'tenure']
)

# Pretty print results
pretty_print_results(results)

### Method 2: Using R-style formula

In [ ]:
# OLS with formula notation
results2 = run_ols_formula('wage ~ educ + exper + tenure + I(exper**2)', data=df)
pretty_print_results(results2)

## 6. Diagnostic Tests

### Heteroskedasticity Tests

In [ ]:
check_heteroskedasticity(results)

### Multicollinearity Check (VIF)

In [ ]:
check_multicollinearity(df, ['educ', 'exper', 'tenure'])

### Autocorrelation Test (Durbin-Watson)

In [ ]:
durbin_watson_test(results)

## 7. Residual Diagnostics

In [ ]:
plot_residuals(results)

## 8. Robust Standard Errors

If you detect heteroskedasticity, use robust standard errors:

In [ ]:
# OLS with robust standard errors
results_robust = run_ols(
    data=df,
    dependent='wage',
    independent=['educ', 'exper', 'tenure'],
    robust=True
)

pretty_print_results(results_robust)

## 9. Advanced: Bootstrap (GPU-Accelerated)

For computationally intensive tasks like bootstrap, you can optionally use GPU acceleration.

**Note:** This requires GPU setup (see README.md)

In [ ]:
# Uncomment if you have GPU set up

# from utils import bootstrap_ols, plot_bootstrap_distribution

# # Run bootstrap (will use GPU if available)
# bootstrap_results = bootstrap_ols(
#     data=df,
#     dependent='wage',
#     independent=['educ', 'exper', 'tenure'],
#     n_iterations=1000,
#     use_gpu=check_gpu_available()
# )

# # Plot bootstrap distribution for education coefficient
# plot_bootstrap_distribution(bootstrap_results, 'educ')

## 10. Tips for Wooldridge Exercises

### Common Workflow:

1. **Load Data**: Use `load_wooldridge_data()`
2. **Explore**: Use `summary_statistics()` and `plot_correlation_matrix()`
3. **Run Regression**: Use `run_ols()` or `run_ols_formula()`
4. **Check Diagnostics**: 
   - `check_heteroskedasticity()`
   - `check_multicollinearity()`
   - `plot_residuals()`
5. **Interpret**: Write your interpretations in markdown cells

### Pro Tips:

- Use formula notation for complex transformations: `'y ~ x + I(x**2) + np.log(z)'`
- Save important plots: `plot_residuals(results, save_path='../outputs/residuals_ch2.png')`
- For large datasets or intensive computations, consider GPU acceleration
- Keep your notebook organized with clear markdown sections
- Document your interpretations as you go

---

## Next Steps

1. Download datasets using `python download_data.py`
2. Copy `chapter_template.ipynb` for each chapter
3. Start solving exercises!

Happy econometrics! 📊

---

## Debugging Data Loading Issues

This section investigates how missing values are handled when loading Wooldridge datasets from Excel files.

In [3]:
# Load BWGHT dataset to investigate missing value handling
bwght = load_wooldridge_data('bwght')
print("BWGHT dataset loaded for debugging")
print(f"Shape: {bwght.shape}")
print(f"Columns: {list(bwght.columns)}")

Loaded bwght from d:\Users\achyu\git\learnStats\Econometrics\Practice\notebooks\..\data\bwght.xls
Shape: (1388, 14)
BWGHT dataset loaded for debugging
Shape: (1388, 14)
Columns: ['faminc', 'cigtax', 'cigprice', 'bwght', 'fatheduc', 'motheduc', 'parity', 'male', 'white', 'cigs', 'lbwght', 'bwghtlbs', 'packs', 'lfaminc']


In [4]:
# Investigate raw Excel data vs processed data for missing values
import pandas as pd
from pathlib import Path

excel_path = Path("../data/bwght.xls")
print("=== Raw Excel Data Investigation ===")

# Read with different parameters to see what's happening
raw_df = pd.read_excel(excel_path, header=None)
print(f"Raw shape: {raw_df.shape}")

# Look at the fatheduc column (should be column 4 based on our mapping)
fatheduc_col_index = 4  # fatheduc is 5th column (0-indexed = 4)
print(f"\nColumn {fatheduc_col_index} (fatheduc) - first 20 values:")
print(raw_df.iloc[:20, fatheduc_col_index].tolist())

print(f"\nUnique values in column {fatheduc_col_index} (showing first 15):")
print(raw_df.iloc[:, fatheduc_col_index].value_counts(dropna=False).head(15))

print(f"\nData type of column {fatheduc_col_index}: {raw_df.iloc[:, fatheduc_col_index].dtype}")

# Check for zeros specifically
zeros_count = (raw_df.iloc[:, fatheduc_col_index] == 0).sum()
print(f"Number of zeros in fatheduc column: {zeros_count}")

# Compare with our processed data
print(f"\n=== Comparison with processed data ===")
print(f"Zeros in processed bwght['fatheduc']: {(bwght['fatheduc'] == 0).sum()}")
print(f"Missing in processed bwght['fatheduc']: {bwght['fatheduc'].isna().sum()}")
print(f"Total observations: {len(bwght)}")
print(f"Valid (non-zero, non-missing): {len(bwght) - (bwght['fatheduc'] == 0).sum() - bwght['fatheduc'].isna().sum()}")

# Check if zeros should be treated as missing values
print(f"\n=== Hypothesis: Zeros are actually missing values ===")
# Exclude zeros to see if we get the expected 1,192 observations
non_zero_fatheduc = bwght[bwght['fatheduc'] != 0]['fatheduc']
print(f"Non-zero observations: {len(non_zero_fatheduc)}")
print(f"Average excluding zeros: {non_zero_fatheduc.mean():.2f}")
print(f"Missing after excluding zeros: {non_zero_fatheduc.isna().sum()}")

=== Raw Excel Data Investigation ===
Raw shape: (1388, 14)

Column 4 (fatheduc) - first 20 values:
[12, 6, '.', 12, 14, 12, 16, 12, 12, 16, 12, 16, '.', 12, 7, 13, 18, '.', 16, '.']

Unique values in column 4 (showing first 15):
4
12    443
.     196
16    189
14    115
18     97
13     87
11     64
10     49
15     43
17     32
8      22
9      17
6      10
7      10
5       4
Name: count, dtype: int64

Data type of column 4: object
Number of zeros in fatheduc column: 0

=== Comparison with processed data ===
Zeros in processed bwght['fatheduc']: 0
Missing in processed bwght['fatheduc']: 0
Total observations: 1388
Valid (non-zero, non-missing): 1388

=== Hypothesis: Zeros are actually missing values ===
Non-zero observations: 1388


TypeError: unsupported operand type(s) for +: 'int' and 'str'